# Mesa-LLM · Plantilla Colab

**Actividad práctica · LLMs en Agent-Based Models · 35 minutos**

Una plantilla mínima y self-contained de un mercado simulado donde **N agentes consumidores** deciden con un LLM si comprar, esperar o negociar.

- ✅ Corre en Colab gratis (CPU).
- ✅ Funciona **sin API key** (modo `mock` determinístico).
- ✅ Funciona con **OpenAI** (`OPENAI_API_KEY`) o **Anthropic** (`ANTHROPIC_API_KEY`).

## Plan

1. Setup (1 celda)
2. Cliente LLM unificado
3. Prompt template (editable)
4. Modelo Mesa: agente + market
5. Correr la simulación
6. Analizar resultados
7. **Tu turno**: cambiar personas / prompt / parámetros

## 1. Setup

In [ ]:
%pip install -q 'mesa>=3.5,<4' matplotlib numpy requests

In [ ]:
import os, json, hashlib, random, re, requests
from collections import Counter, defaultdict
from dataclasses import dataclass
from typing import Any, Optional

import numpy as np
import matplotlib.pyplot as plt
import mesa

print('mesa version :', mesa.__version__)

# (Opcional) Setear API key de OpenAI/Anthropic.
# Si no ponés ninguna, el cliente cae en el mock automáticamente.
# os.environ['OPENAI_API_KEY']    = 'sk-...'
# os.environ['ANTHROPIC_API_KEY'] = 'sk-ant-...'

## 2. Cliente LLM unificado (con fallback offline)

Mismo módulo que en `llm/client.py` del repo, pero inline para Colab.

In [ ]:
# --- Mock LLM determinístico --------------------------------------------- #
_KEYWORDS_BUY      = ('impulsivo', 'fashion', 'gourmet', 'hedonista', 'ostentoso')
_KEYWORDS_SAVE     = ('ahorrativo', 'minimalista', 'racional', 'frugal', 'esceptico', 'escéptico')
_KEYWORDS_NEGOTIATE = ('negociador', 'regateador', 'experto', 'comerciante')

def _seed_from_prompt(prompt: str, salt: int = 0) -> int:
    h = hashlib.sha1(f'{salt}|{prompt}'.encode('utf-8')).hexdigest()
    return int(h[:8], 16)

def fake_consumer(prompt: str, salt: int = 0) -> dict:
    rng = random.Random(_seed_from_prompt(prompt, salt))
    p = prompt.lower()
    if any(k in p for k in _KEYWORDS_BUY):
        bias = 'buy'
    elif any(k in p for k in _KEYWORDS_NEGOTIATE):
        bias = 'negotiate'
    elif any(k in p for k in _KEYWORDS_SAVE):
        bias = 'save'
    else:
        bias = 'neutral'

    m = re.search(r'Tu presupuesto[^\n]*?\$?\s*(\d+)', prompt, re.IGNORECASE)
    presupuesto = int(m.group(1)) if m else 100
    ids = [int(x) for x in re.findall(r'id=(\d+)', prompt)] or list(range(5))

    if bias == 'buy':
        accion = rng.choices(['comprar', 'esperar', 'negociar'], weights=[7,2,1])[0]
    elif bias == 'save':
        accion = rng.choices(['comprar', 'esperar', 'negociar'], weights=[2,6,2])[0]
    elif bias == 'negotiate':
        accion = rng.choices(['comprar', 'esperar', 'negociar'], weights=[3,2,5])[0]
    else:
        accion = rng.choices(['comprar', 'esperar', 'negociar'], weights=[4,4,2])[0]

    if presupuesto < 30 and accion == 'comprar':
        accion = rng.choice(['esperar', 'negociar'])

    pid = rng.choice(ids) if accion in ('comprar', 'negociar') else None
    razon = {
        'comprar' : ['precio razonable', 'lo necesito ahora', 'recomendado'],
        'esperar' : ['espero que baje', 'no me convence', 'no urgente'],
        'negociar': ['está caro', 'busco descuento', 'tengo otra oferta'],
    }[accion]
    emo = {
        'buy'    : ['entusiasmado', 'esperanzado', 'satisfecho'],
        'save'   : ['neutro', 'calmo', 'expectante'],
        'negotiate': ['atento', 'expectante'],
        'neutral': ['neutro', 'expectante']
    }[bias]
    return {'accion': accion, 'producto_id': pid,
            'razon': rng.choice(razon),
            'estado_emocional': rng.choice(emo)}

def mock_respond(prompt: str, salt: int = 0) -> str:
    return json.dumps(fake_consumer(prompt, salt=salt), ensure_ascii=False)


# --- Cliente unificado --------------------------------------------------- #
@dataclass
class LLMResponse:
    text: str
    parsed: Optional[dict]
    provider: str
    model: Optional[str] = None

class LLMClient:
    def __init__(self, provider='auto', model=None, temperature=0.7, timeout=30.0):
        self.temperature = temperature; self.timeout = timeout
        if provider == 'auto':
            if os.environ.get('OPENAI_API_KEY'):       provider = 'openai'
            elif os.environ.get('ANTHROPIC_API_KEY'):  provider = 'anthropic'
            else:                                       provider = 'mock'
        self.provider = provider
        self.model = model or {
            'openai':    os.environ.get('OPENAI_MODEL', 'gpt-4o-mini'),
            'anthropic': os.environ.get('ANTHROPIC_MODEL', 'claude-3-5-haiku-latest'),
            'mock':      'mock-rule-based',
        }.get(provider)

    def complete_json(self, prompt: str, salt: int = 0) -> LLMResponse:
        if self.provider == 'mock':
            text = mock_respond(prompt, salt=salt)
        elif self.provider == 'openai':
            r = requests.post(
                'https://api.openai.com/v1/chat/completions',
                headers={'Authorization': f"Bearer {os.environ['OPENAI_API_KEY']}",
                         'Content-Type': 'application/json'},
                json={'model': self.model, 'temperature': self.temperature,
                      'response_format': {'type':'json_object'},
                      'messages':[
                          {'role':'system','content':'Sos un agente de simulación. Respondés sólo JSON válido.'},
                          {'role':'user','content':prompt}]},
                timeout=self.timeout)
            r.raise_for_status()
            text = r.json()['choices'][0]['message']['content']
        elif self.provider == 'anthropic':
            r = requests.post(
                'https://api.anthropic.com/v1/messages',
                headers={'x-api-key': os.environ['ANTHROPIC_API_KEY'],
                         'anthropic-version': '2023-06-01',
                         'Content-Type': 'application/json'},
                json={'model': self.model, 'max_tokens': 512,
                      'temperature': self.temperature,
                      'messages':[{'role':'user',
                                   'content':prompt+'\n\nRespondé únicamente con JSON válido.'}]},
                timeout=self.timeout)
            r.raise_for_status()
            text = r.json()['content'][0]['text']
        else:
            raise ValueError(self.provider)

        try:
            parsed = json.loads(text)
        except Exception:
            m = re.search(r'\{.*\}', text, re.DOTALL)
            parsed = json.loads(m.group(0)) if m else None
        return LLMResponse(text=text, parsed=parsed,
                           provider=self.provider, model=self.model)

client = LLMClient(provider='auto')
print(f"Provider activo : {client.provider} (model={client.model})")

## 3. Prompt template

**Editá esta celda** para cambiar el comportamiento de los agentes.

In [ ]:
PROMPT_TEMPLATE = """Sos un consumidor en un mercado simulado.

Personalidad: {persona}
Memoria reciente: {memoria}
Situación actual: {estado}
Productos disponibles: {productos}
Tu presupuesto: ${presupuesto}

Decidí qué hacer en este turno y por qué.
Respondé EXCLUSIVAMENTE como JSON con la siguiente forma:

{{
    "accion": "comprar" | "esperar" | "negociar",
    "producto_id": int o null,
    "razon": string corto,
    "estado_emocional": string
}}

No agregues texto fuera del JSON.
"""

def render(template, **kwargs):
    out = template
    for k, v in kwargs.items():
        out = out.replace('{' + k + '}', str(v))
    return out

## 4. Modelo Mesa

In [ ]:
PERSONAS = [
    'ahorrativo, busca siempre el precio más bajo',
    'impulsivo, fashion-victim, sigue las modas',
    'escéptico, desconfía del marketing',
    'racional, evalúa precio-calidad fríamente',
    'negociador, profesional del regateo',
    'gourmet, valora calidad por sobre precio',
    'minimalista, sólo lo necesario',
    'ostentoso, le gusta mostrar lo que tiene',
]

PRODUCTS = [
    {'id': 0, 'nombre': 'Pan integral',     'precio_base': 8},
    {'id': 1, 'nombre': 'Café especialidad','precio_base': 25},
    {'id': 2, 'nombre': 'Smartphone',       'precio_base': 350},
    {'id': 3, 'nombre': 'Camisa Aurora',    'precio_base': 45},
    {'id': 4, 'nombre': 'Auto usado',       'precio_base': 8000},
]

class ConsumerAgent(mesa.Agent):
    def __init__(self, model, persona, presupuesto):
        super().__init__(model)
        self.persona = persona
        self.presupuesto = presupuesto
        self.gastado = 0
        self.memoria = []
        self.acciones = []

    def step(self):
        productos_str = ', '.join(
            f"id={p['id']} {p['nombre']} (${self.model.prices[p['id']]})"
            for p in PRODUCTS)
        prompt = render(PROMPT_TEMPLATE,
                        persona=self.persona,
                        memoria='; '.join(self.memoria[-3:]) or '(sin memoria)',
                        estado=f'step {self.model.steps_run}',
                        productos=productos_str,
                        presupuesto=self.presupuesto - self.gastado)
        resp = self.model.llm.complete_json(prompt,
                                            salt=self.unique_id*31 + self.model.steps_run)
        d = resp.parsed or {'accion':'esperar','producto_id':None,
                            'razon':'no parsable','estado_emocional':'neutro'}
        self._apply(d)

    def _apply(self, d):
        accion, pid = d.get('accion','esperar'), d.get('producto_id')
        if accion == 'comprar' and isinstance(pid, int) and 0 <= pid < len(PRODUCTS):
            price = self.model.prices[pid]
            if price <= self.presupuesto - self.gastado:
                self.gastado += price
                self.model.demand[pid] += 1
                self.acciones.append('comprar')
                self.memoria.append(f"compré {PRODUCTS[pid]['nombre']} a ${price}")
                return
            accion = 'esperar'
        if accion == 'negociar' and isinstance(pid, int) and 0 <= pid < len(PRODUCTS):
            self.model.proposed_discounts[pid] += 1
            self.acciones.append('negociar'); return
        self.acciones.append('esperar')

class MarketModel(mesa.Model):
    def __init__(self, n_agents=12, provider='auto', model_name=None, seed=42):
        super().__init__(rng=seed)
        self.steps_run = 0
        self.prices = {p['id']: p['precio_base'] for p in PRODUCTS}
        self.demand = Counter()
        self.proposed_discounts = Counter()
        self.llm = LLMClient(provider=provider, model=model_name)
        self.history = {p['id']: [] for p in PRODUCTS}
        for i in range(n_agents):
            ag = ConsumerAgent(self,
                               persona=PERSONAS[i % len(PERSONAS)],
                               presupuesto=self.random.choice([60,90,120,200,500,1500,9000]))
            ag.unique_id = i

    def step(self):
        self.demand.clear(); self.proposed_discounts.clear()
        self.agents.shuffle_do('step')
        self.steps_run += 1
        for pid in self.prices:
            base = PRODUCTS[pid]['precio_base']
            change = 0.04*self.demand[pid] - 0.02*self.proposed_discounts[pid]
            new = self.prices[pid]*(1+change)
            new = 0.7*new + 0.3*base
            self.prices[pid] = max(1, int(round(new)))
            self.history[pid].append(self.prices[pid])

## 5. Correr la simulación

In [ ]:
STEPS = 15
model = MarketModel(n_agents=12, provider='auto')
print(f"Provider efectivo: {model.llm.provider}\n")

for t in range(1, STEPS + 1):
    model.step()
    if t in (1, 5, 10, STEPS):
        actions = Counter(a.acciones[-1] for a in model.agents)
        print(f"step {t:2d}  {dict(actions)}  prices={model.prices}")

## 6. Analizar resultados

In [ ]:
actions_pers = defaultdict(Counter)
spend_pers = defaultdict(int)
for ag in model.agents:
    key = ag.persona.split(',')[0]
    spend_pers[key] += ag.gastado
    for a in ag.acciones:
        actions_pers[key][a] += 1

fig, axes = plt.subplots(1, 3, figsize=(14, 4.5))
keys = sorted(actions_pers)
actions_list = ['comprar', 'negociar', 'esperar']
colors = {'comprar':'#172186', 'negociar':'#FD8204', 'esperar':'#cccccc'}
bottom = np.zeros(len(keys))
for a in actions_list:
    vals = np.array([actions_pers[k].get(a,0) for k in keys])
    axes[0].barh(keys, vals, left=bottom, label=a, color=colors[a])
    bottom += vals
axes[0].set_title('Acciones por persona'); axes[0].legend(loc='lower right')

k2 = sorted(spend_pers, key=lambda k: spend_pers[k], reverse=True)
axes[1].barh(k2, [spend_pers[k] for k in k2], color='#172186')
axes[1].set_title('Gasto total ($)'); axes[1].invert_yaxis()

for pid in model.history:
    axes[2].plot(model.history[pid], label=PRODUCTS[pid]['nombre'][:14])
axes[2].set_yscale('log'); axes[2].set_title('Precios')
axes[2].legend(fontsize=8)
plt.tight_layout()
plt.show()

## 7. Tu turno

**Cosas para experimentar (ordenadas por dificultad creciente):**

1. **Cambiar `PERSONAS`** — agregá vos un par y mirá cómo cambia el equilibrio.
2. **Editar `PROMPT_TEMPLATE`** — agregá una restricción (ej: "odiás los productos de Aurora") y ejecutá de nuevo.
3. **Subir a 30 agentes y 30 steps** — ¿se estabiliza la distribución?
4. **Activar provider real**: descomentá `os.environ['OPENAI_API_KEY']` arriba y compará con el mock.
5. **Memoria compartida**: hacé que los agentes vean compras de otros (`pasale al prompt una lista global`).
6. **Tipos de agentes mixtos**: 80% mock + 20% LLM real, ¿se nota la diferencia?

**Para Opción B (avanzada)** mirá `examples/opcion_b_trafico.py` en el repo.